In [1]:
# Google Colab Only
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval maite-datasets torchvision
except Exception:
    pass

In [2]:
from collections.abc import Callable

import numpy as np
from maite_datasets.image_classification import MilitaryVehicles
from torchvision.models import ResNet18_Weights, resnet18

from dataeval import Embeddings
from dataeval.core import divergence_fnn, divergence_mst
from dataeval.data import Limit, Shuffle, View
from dataeval.extractors import TorchExtractor

In [3]:
SAMPLE = 800

train_ds = MilitaryVehicles("./data", image_set="train", download=True, lazy=True)
test_ds = MilitaryVehicles("./data", image_set="test", download=True, lazy=True)

weights = ResNet18_Weights.DEFAULT
extractor = TorchExtractor(
    resnet18(weights=weights, progress=False), transforms=weights.transforms(), layer_name="avgpool"
)


def embed(dataset: MilitaryVehicles, size: int) -> np.ndarray:
    """Embed a random sample of the dataset. The extractor holds no state, so one instance serves every dataset."""
    return np.asarray(Embeddings(View(dataset, [Shuffle(seed=0), Limit(size)]), extractor=extractor, batch_size=64))


train_embeddings = embed(train_ds, 2 * SAMPLE)
test_embeddings = embed(test_ds, SAMPLE)

In [4]:
data_a = train_embeddings[:SAMPLE]
data_b = train_embeddings[SAMPLE:]

In [5]:
div = divergence_fnn(data_a, data_b)
print(div)

{'divergence': np.float64(0.013749999999999929), 'errors': np.int64(789)}


In [6]:
reference = []
for seed in range(8):
    order = np.random.default_rng(seed).permutation(len(train_embeddings))
    reference.append(divergence_fnn(train_embeddings[order[:SAMPLE]], train_embeddings[order[SAMPLE:]])["divergence"])

print(f"Two samples of the training set, 8 splits: mean {np.mean(reference):.3f}, largest {max(reference):.3f}")

Two samples of the training set, 8 splits: mean 0.012, largest 0.050


In [7]:
train_test = divergence_fnn(data_a, test_embeddings)["divergence"]
print(f"Train against test: {train_test:.3f}")

Train against test: 0.016


In [8]:
def dim(factor: float) -> Callable[[np.ndarray], np.ndarray]:
    """Return a transform that scales every pixel by the factor."""
    return lambda image: (image * factor).astype(np.uint8)


dimmed = {}
for factor in (0.9, 0.7, 0.5):
    dim_ds = MilitaryVehicles("./data", image_set="test", download=True, lazy=True, transforms=dim(factor))
    dim_embeddings = embed(dim_ds, SAMPLE)
    dimmed[factor] = {
        "fnn": divergence_fnn(data_a, dim_embeddings)["divergence"],
        "mst": divergence_mst(data_a, dim_embeddings)["divergence"],
    }
    print(f"Test images at {factor:.0%} brightness: fnn {dimmed[factor]['fnn']:.3f}, mst {dimmed[factor]['mst']:.3f}")

Test images at 90% brightness: fnn 0.046, mst 0.035


Test images at 70% brightness: fnn 0.179, mst 0.176


Test images at 50% brightness: fnn 0.384, mst 0.384


In [9]:
# TEST ASSERTION CELL ###
assert train_embeddings.shape == (2 * SAMPLE, 512)
assert test_embeddings.shape == (SAMPLE, 512)
assert max(reference) < 0.08  # "about 0.05"
assert train_test <= max(reference) + 0.02  # "as close as two training samples"
assert abs(dimmed[0.9]["fnn"] - max(reference)) < 0.03  # "close to the largest reference value"
assert abs(dimmed[0.7]["fnn"] - 0.18) < 0.02
assert abs(dimmed[0.5]["fnn"] - 0.38) < 0.02
assert dimmed[0.7]["fnn"] > 3 * max(reference)
assert dimmed[0.9]["fnn"] < dimmed[0.7]["fnn"] < dimmed[0.5]["fnn"] < 0.5  # "far from 1"
assert all(abs(v["fnn"] - v["mst"]) < 0.02 for v in dimmed.values())  # "the estimators agree"
assert (
    0.45 < np.asarray(dim_ds[0][0]).mean() / np.asarray(test_ds[0][0]).mean() < 0.55
)  # the dimming reaches the images